# 🐍 Zahhak-snakehandler on Google Colab

<a href="https://colab.research.google.com/github/kheradafrouz/Zahhak-snakehandler/blob/main/notebooks/Zahhak_snakehandler_Colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

Train a Double-DQN agent that teaches itself Snake, then evaluate it against a random baseline and watch it play.
Works on **CPU, GPU (CUDA) and TPU** runtimes, and keeps your **best model on Google Drive** so it survives disconnects.

| Runtime (`Runtime > Change runtime type`) | Device | Notes |
|---|---|---|
| CPU | `cpu` | Works everywhere. Fast enough for the default small network. |
| T4 GPU (or better) | `cuda` | Pairs well with the `gpu` preset (bigger network and batches). |
| TPU | `tpu` | Best-effort via PyTorch/XLA. This tiny network at batch size 1 for action selection is usually **slower** on a TPU than on CPU. Use the benchmark cell to check. |

**How to use:** pick a runtime, edit the settings cell, then `Runtime > Run all`.
If the session disconnects, just run all again: training resumes from Google Drive.

**What is saved to Drive** (`MyDrive/<DRIVE_FOLDER>/checkpoints/`):
`Zahhak-snakehandler_best.pt` (best greedy-evaluation score), `..._best_meta.json` (its scores), and `..._latest.pt` (for resuming).

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
DEVICE = "auto"            #@param ["auto", "cpu", "cuda", "tpu"]
PRESET = "auto"            #@param ["auto", "cpu", "gpu"]
EPISODES = 3000            #@param {type:"integer"}
SEED = 42                  #@param {type:"integer"}
RESUME = True              #@param {type:"boolean"}
EVAL_EVERY = 250           #@param {type:"integer"}
EVAL_EPISODES = 20         #@param {type:"integer"}
CODE_SOURCE = "github"     #@param ["github", "upload", "drive"]
GITHUB_REPO = "https://github.com/kheradafrouz/Zahhak-snakehandler.git"  #@param {type:"string"}
DRIVE_FOLDER = "Zahhak-snakehandler"  #@param {type:"string"}

# EPISODES = number of episodes to train in THIS run (set 0 to skip training and only evaluate).
# CODE_SOURCE: "github" clones GITHUB_REPO; "upload" asks you to upload Zahhak-snakehandler.zip;
#              "drive" unzips MyDrive/<DRIVE_FOLDER>/Zahhak-snakehandler.zip.

## 2. Connect Google Drive

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK = Path("/content")
else:
    DRIVE_ROOT = Path.cwd() / "drive_fallback" / DRIVE_FOLDER
    WORK = Path.cwd()
    print("Not running in Colab: using a local folder instead of Google Drive.")

CKPT_DIR = DRIVE_ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
probe = CKPT_DIR / ".write_test"
probe.write_text("ok")
probe.unlink()
print("Checkpoints are saved to:", CKPT_DIR)

## 3. Get the code

In [ ]:
REPO_DIR = WORK / "Zahhak-snakehandler"

def fetch_code():
    if (REPO_DIR / "zahhak_snakehandler").is_dir():
        print("Code already present at", REPO_DIR)
        return
    if CODE_SOURCE == "github":
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(REPO_DIR)], check=True)
    elif CODE_SOURCE == "upload":
        from google.colab import files
        print("Select Zahhak-snakehandler.zip ...")
        uploaded = files.upload()
        shutil.unpack_archive(next(iter(uploaded)), str(WORK))
    elif CODE_SOURCE == "drive":
        shutil.unpack_archive(str(DRIVE_ROOT / "Zahhak-snakehandler.zip"), str(WORK))
    if not (REPO_DIR / "zahhak_snakehandler").is_dir():
        raise FileNotFoundError(f"Expected the package at {REPO_DIR}/zahhak_snakehandler. "
                                "Check CODE_SOURCE / GITHUB_REPO, or that the zip has a Zahhak-snakehandler/ root folder.")

fetch_code()
sys.path.insert(0, str(REPO_DIR))
import zahhak_snakehandler
print("Zahhak-snakehandler", zahhak_snakehandler.__version__)
# Note: Colab already ships PyTorch/NumPy. We deliberately do NOT reinstall torch (it would replace the
# GPU/TPU build). Pygame is not needed here: this notebook renders plots and GIFs instead of a window.

## 4. Pick the device (CPU / GPU / TPU)

In [ ]:
import torch

TPU_HELP = (
    "TPU requested but PyTorch/XLA is not ready.\n"
    "1) Runtime > Change runtime type > choose a TPU.\n"
    "2) If torch_xla is missing, run this in a new cell, then Runtime > Restart session and run all again:\n"
    "   !pip install torch~=2.5.0 torch_xla[tpu]~=2.5.0 -f https://storage.googleapis.com/libtpu-releases/index.html\n"
    "3) Or use a GPU / CPU runtime. For this small network they are usually faster anyway."
)

def looks_like_tpu():
    return bool(list(Path("/dev").glob("accel*")) or os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"))

def tpu_ready():
    try:
        os.environ.setdefault("PJRT_DEVICE", "TPU")
        import torch_xla.core.xla_model as xm
        xm.xla_device()
        return True
    except Exception:
        return False

def pick_device(requested):
    if requested == "cpu":
        return "cpu"
    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but no GPU found. Runtime > Change runtime type > GPU.")
        return "cuda"
    if requested == "tpu":
        if not tpu_ready():
            raise RuntimeError(TPU_HELP)
        return "tpu"
    if torch.cuda.is_available():            # auto
        return "cuda"
    if looks_like_tpu() and tpu_ready():
        return "tpu"
    return "cpu"

DEV = pick_device(DEVICE)
print("Using device:", DEV, "-", torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 5. Build the trainer (and resume from Drive if possible)

In [ ]:
from dataclasses import replace
from zahhak_snakehandler.config import Config
from zahhak_snakehandler.train import Trainer, log_line
from zahhak_snakehandler.evaluate import run_episodes, compare, print_report
from zahhak_snakehandler.agent import load_agent

preset = PRESET if PRESET != "auto" else ("gpu" if DEV in ("cuda", "tpu") else "cpu")
base = Config.gpu_preset() if preset == "gpu" else Config()
cfg = replace(base, device=DEV, seed=SEED, checkpoint_dir=str(CKPT_DIR),
              eval_every=EVAL_EVERY, eval_episodes=EVAL_EPISODES)

latest_path = Path(cfg.checkpoint_path("latest"))
best_path = Path(cfg.checkpoint_path("best"))
meta_path = CKPT_DIR / f"{cfg.model_name}_best_meta.json"

# Not resuming but files exist: archive them so nothing on Drive is silently overwritten.
if not RESUME and (latest_path.exists() or best_path.exists()):
    archive = CKPT_DIR / f"archive_{time.strftime('%Y%m%d_%H%M%S')}"
    archive.mkdir()
    for p in CKPT_DIR.glob(f"{cfg.model_name}_*"):
        shutil.move(str(p), str(archive / p.name))
    print("Existing Drive files moved to", archive)

trainer = Trainer(cfg)
if RESUME and latest_path.exists():
    trainer.load(str(latest_path))
    print(f"Resumed from Drive: episode {trainer.episode}, epsilon {trainer.agent.epsilon:.3f}, "
          f"best eval so far {trainer.best_eval:.2f}")
else:
    print("Starting a fresh run.")
print(f"{cfg.model_name} | device {trainer.agent.device} | preset {preset} | hidden {cfg.hidden_sizes} | "
      f"batch {cfg.batch_size} | seed {cfg.seed}")

## 6. Train
Live plots refresh during training. **The best model is saved to Drive whenever the greedy-evaluation score improves**
(evaluation uses epsilon = 0 on a separate seeded environment, so it is not the noisy training score).
Press the stop button to interrupt safely: the latest checkpoint is written first.

In [ ]:
import collections
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

TMP = WORK / "_ckpt_tmp.pt"

def save_ckpt(kind):
    """Write locally first, then copy to Drive via a temp name so a disconnect never leaves a half-written file."""
    final = Path(cfg.checkpoint_path(kind))
    trainer.save(str(TMP))
    final.parent.mkdir(parents=True, exist_ok=True)
    staged = final.with_name(final.name + ".tmp")
    shutil.copy2(TMP, staged)
    os.replace(staged, final)
    return final

logs = collections.deque(maxlen=14)
eval_hist = []

def refresh():
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    sc = np.asarray(trainer.scores, dtype=float)
    if len(sc):
        ax[0].plot(sc, alpha=0.3, label="episode score")
        if len(sc) >= 20:
            ax[0].plot(range(19, len(sc)), np.convolve(sc, np.ones(20) / 20, "valid"), label="20-episode average")
        ax[0].legend()
    ax[0].set(title="Training score (epsilon-greedy)", xlabel="episode")
    if eval_hist:
        e = np.asarray(eval_hist)
        ax[1].plot(e[:, 0], e[:, 1], "o-")
    ax[1].set(title="Greedy evaluation (average score)", xlabel="episode")
    plt.tight_layout()
    plt.show()
    print("\n".join(logs))

start_ep = trainer.episode
target = start_ep + EPISODES
t0, s0 = time.perf_counter(), trainer.total_steps
try:
    while trainer.episode < target:
        if not trainer.step():
            continue
        ep = trainer.episode
        refresh_now = False
        if ep % cfg.log_every == 0:
            now = time.perf_counter()
            logs.append(log_line(trainer, (trainer.total_steps - s0) / max(now - t0, 1e-9)))
            t0, s0 = now, trainer.total_steps
            refresh_now = True
        if ep % cfg.eval_every == 0:
            ev = run_episodes(cfg, lambda s: trainer.agent.act(s, greedy=True), cfg.eval_episodes, cfg.seed + 999)
            eval_hist.append((ep, ev["avg_score"]))
            line = (f"  [EVAL greedy, {cfg.eval_episodes} eps] avg {ev['avg_score']:.2f} | max {ev['max_score']} "
                    f"| survival {ev['avg_steps']:.0f} steps")
            if ev["avg_score"] > trainer.best_eval:
                trainer.best_eval = ev["avg_score"]
                path = save_ckpt("best")
                meta_path.write_text(json.dumps({
                    "episode": ep, "total_steps": trainer.total_steps, "device": str(trainer.agent.device),
                    "preset": preset, "seed": cfg.seed, "eval_episodes": cfg.eval_episodes,
                    "eval_avg_score": ev["avg_score"], "eval_max_score": ev["max_score"],
                    "eval_avg_steps": ev["avg_steps"], "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                }, indent=2))
                line += f"  -> NEW BEST saved to Drive ({path.name})"
            logs.append(line)
            refresh_now = True
        if ep % cfg.save_every_episodes == 0:
            save_ckpt("latest")
        if refresh_now:
            refresh()
except KeyboardInterrupt:
    print("Interrupted: saving the latest checkpoint to Drive ...")
finally:
    if trainer.episode > start_ep:
        save_ckpt("latest")
        print(f"Saved latest checkpoint at episode {trainer.episode}: {latest_path}")
    else:
        print("No new episodes were trained; checkpoints on Drive left untouched.")

## 7. Evaluate the best model from Drive
This loads the saved best checkpoint (not the in-memory agent) and compares it with a random-action baseline.
These are **evaluation** numbers (no exploration); they are the ones that tell you whether the agent learned.

In [ ]:
if not best_path.exists():
    print("No best checkpoint on Drive yet (a best model is saved at the first evaluation, "
          f"i.e. after {cfg.eval_every} episodes). Train longer or lower EVAL_EVERY.")
else:
    agent, ckpt = load_agent(str(best_path), cfg)
    print(f"Loaded {best_path.name} (saved at episode {ckpt['trainer']['episode']}) onto {agent.device}")
    if meta_path.exists():
        print(meta_path.read_text())
    res = compare(agent, agent.cfg, episodes=100, seed=cfg.seed + 2024)
    print_report(res)

## 8. Watch the trained agent play

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, display
from zahhak_snakehandler.snake_env import SnakeEnv

CELL = 24

def draw(env):
    w, h = env.w * CELL, env.h * CELL
    img = Image.new("RGB", (w, h), (18, 20, 26))
    d = ImageDraw.Draw(img)
    for i in range(env.w + 1):
        d.line([(i * CELL, 0), (i * CELL, h)], fill=(30, 33, 42))
    for j in range(env.h + 1):
        d.line([(0, j * CELL), (w, j * CELL)], fill=(30, 33, 42))
    fx, fy = env.food
    d.ellipse([fx * CELL + 4, fy * CELL + 4, (fx + 1) * CELL - 4, (fy + 1) * CELL - 4], fill=(230, 70, 80))
    for k, (x, y) in enumerate(env.snake):
        d.rounded_rectangle([x * CELL + 2, y * CELL + 2, (x + 1) * CELL - 2, (y + 1) * CELL - 2],
                            radius=6, fill=(130, 240, 160) if k == 0 else (60, 190, 110))
    return img

def rollout(agent, seed, max_steps=600):
    env = SnakeEnv(agent.cfg, seed=seed)
    s = env.reset()
    frames, done = [draw(env)], False
    while not done and len(frames) < max_steps:
        s, _, done, _ = env.step(agent.act(s, greedy=True))
        frames.append(draw(env))
    return env.score, frames

if not best_path.exists():
    print("Train first so a best model exists on Drive.")
else:
    agent, _ = load_agent(str(best_path), cfg)
    runs = [rollout(agent, cfg.seed + 500 + i) for i in range(5)]
    scores = [r[0] for r in runs]
    score, frames = max(runs, key=lambda r: r[0])
    gif = WORK / "play.gif"
    frames[0].save(gif, save_all=True, append_images=frames[1:], duration=70, loop=0)
    print(f"5 greedy rollouts scored {scores}. Showing the highest-scoring one ({score}); "
          "it was picked for display, so use the evaluation table above for honest numbers.")
    display(IPyImage(filename=str(gif)))

## 9. (Optional) Which device is fastest for you?
Runs the same short training workload on CPU and on the selected accelerator. On TPU the first steps include XLA compilation,
so treat that figure as a rough guide.

In [ ]:
def bench(device, steps=4000):
    tr = Trainer(replace(cfg, device=device, seed=1))
    for _ in range(1200):            # warm-up: fills the buffer, starts learning, triggers compilation on XLA
        tr.step()
    t = time.perf_counter()
    for _ in range(steps):
        tr.step()
    return steps / (time.perf_counter() - t)

for d in dict.fromkeys(["cpu", DEV]):
    print(f"{d:>5}: {bench(d):>9,.0f} env steps/s (training, preset={preset})")

## 10. (Optional) Download the best model

In [ ]:
if IN_COLAB and best_path.exists():
    from google.colab import files
    files.download(str(best_path))
else:
    print("Best model on Drive:", best_path)

## Troubleshooting
- **Drive mount fails or asks again:** re-run the Drive cell and allow access. Files live in `MyDrive/Zahhak-snakehandler/checkpoints`.
- **`CUDA requested but no GPU found`:** `Runtime > Change runtime type > T4 GPU`, or set `DEVICE = "auto"`.
- **TPU errors:** see the message from the device cell. TPU support is best-effort; CPU or GPU is usually faster for this network.
- **Hidden-size mismatch on resume:** you changed `PRESET`. Use the same preset, or set `RESUME = False` (old files are archived, not deleted).
- **Free-tier disconnects:** run all cells again. Training resumes from `..._latest.pt`; the best model is never lost.